# 🧲 Example 3: Adding Magnets & Calibrating Tolerances

This tutorial explains how to hollow out hemispheres and add magnet joint cavities to the equatorial mating surface so your split hemispheres snap together perfectly. 

### 🌎 Scientific & Design Context
- **Magnet Joints**: For interactive models (like layered interiors or deep-mantle tomography shells), embedding magnets allows users to easily take the globe apart and assemble it without loose pins or latches.
- **Tolerances**: Because 3D printers squeeze hot plastic slightly outward, holes print smaller than designed. We configure horizontal and vertical tolerances to ensure magnets slide in snugly. We also print a quick **calibration test piece** to tune these settings before committing to a long globe print.

## Step 1: Import Libraries

We import `GlobeModel` and the calibration utility.

In [ ]:
import os
from globe3d import GlobeModel, generate_magnet_test_piece

## Step 2: Prepare a Hollow Base Sphere

We initialize a model with `hollow=True` and setting `inner_ratio=0.5`. This reserves a hollow cavity spanning $50\%$ of the outer radius, leaving a thick wall perfect for housing magnet pockets.

In [ ]:
model_radius_mm = 40.0

model = GlobeModel(
    n_points=6000,
    radius=model_radius_mm,
    hollow=True,
    inner_ratio=0.5,  # thick shell wall for housing magnets
)
print(f"Outer shell: {model.outer.vertices.shape[0]} vertices")
print(f"Inner cavity: {model.inner.vertices.shape[0]} vertices")

## Step 3: Alternative A: Hollowing with Magnet BOSSES (`add_bosses=True`)

If the shell walls are thin, or if you want magnets at precise angles, we configure the library to build support towers (bosses) around each magnet pocket on the inside wall of the cavity. This prevents the holes from breaking through into the hollow center.

In [ ]:
# Configure magnets with bosses
model.configure_magnets(
    diameter=5.0,                  # 5mm disc magnet
    height=2.0,                    # 2mm depth
    n_magnets=3,                   # 3 magnets evenly spaced
    position=0.0,                  # Start angle in degrees
    horizontal_tolerance=0.15,     # 0.15mm radial tolerance
    vertical_tolerance=0.10,       # 0.10mm depth tolerance
    vertical_offset=0.20,          # 0.20mm plastic ceiling thickness
    min_thickness=1.5,             # 1.5mm wall surrounding the void
    add_bosses=True,               # Union support cylinders inside cavity
)

# Generate hemispheres (hollowing and magnet Boolean operations handled automatically)
top_boss, bottom_boss = model.generate_hemispheres(engine='manifold')
print(f"Top boss mesh watertight: {top_boss.is_watertight}")
print(f"Bottom boss mesh watertight: {bottom_boss.is_watertight}")

## Step 4: Alternative B: Hollowing WITHOUT Bosses (Inside Shell)

If you want to keep the interior cavity completely smooth, set `add_bosses=False`. The library will search the equator in 2-degree increments to find coordinates where magnet voids fit entirely inside the natural shell thickness, maximizing spacing symmetry.

In [ ]:
# Reconfigure magnets to sit strictly inside the natural wall thickness
model.configure_magnets(
    diameter=5.0,
    height=2.0,
    n_magnets=3,
    min_magnets=2,
    min_angular_spacing=60.0,
    step_degrees=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    add_bosses=False,
)

top_noboss, bottom_noboss = model.generate_hemispheres(engine='manifold')
print(f"Top no-boss mesh watertight: {top_noboss.is_watertight}")
print(f"Bottom no-boss mesh watertight: {bottom_noboss.is_watertight}")

## Step 5: Generate a Calibration Test Piece

Instead of printing an 8-hour globe to test if your tolerances are correct, we export a quick 10-minute cylinder containing a single magnet pocket. Adjust the tolerances based on how snugly your magnet pushes in!

In [ ]:
os.makedirs('../outputs', exist_ok=True)
test_piece_path = "../outputs/magnet_test_piece.stl"

generate_magnet_test_piece(
    diameter=5.0,
    height=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    output_path=test_piece_path,
)
print(f"Saved calibration test piece to: {test_piece_path}")

## 6. Preview the Model in 3D

Preview the interactive 3D model:

In [ ]:
model.preview()